# 🔬 viMSam MicroSAM Pipeline

This notebook runs the `vimsam_segmenter` package from trainer_app.

The supported interfaces are:
- `vimsam-segmenter` (CLI command)
- `python /content/trainer_app/main.py` (local entry point)


In [ ]:
# ============================================================
# 1. Colab Environment Setup
# ============================================================

# Install condacolab to enable conda/mamba support in Colab.
!pip install -q condacolab

import condacolab

# This installs a conda environment into the Colab runtime.
# Colab may restart the runtime after this cell.
condacolab.install()

In [ ]:
# ============================================================
# 2. Verify Conda Environment
# ============================================================

import condacolab

condacolab.check()

In [ ]:
# ============================================================
# 3. Dependency Installation
# ============================================================

# Remove any existing conda pinning that could conflict with dependency resolution.
!rm -f /usr/local/conda-meta/pinned

# Install NumPy version used by the original environment.
!mamba install -c conda-forge "numpy=1.26.4" -y

# Install MicroSAM and required Python dependencies (Updated PyTorch to >=2.4)
!mamba install -c conda-forge \
    micro_sam \
    "pytorch>=2.4" \
    torchvision \
    imageio \
    imageio-ffmpeg \
    tifffile \
    python-dotenv \
    pandas \
    scikit-image \
    -y

# Install system ffmpeg.
!apt-get update && apt-get install -y ffmpeg


In [ ]:
# ============================================================
# 4. Runtime Verification
# ============================================================
#
# Verifies:
#   - PyTorch version
#   - CUDA/GPU availability
#   - NumPy version
#   - ffmpeg availability
# ============================================================

import os
import shutil
import subprocess

import torch
import numpy as np

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU is not available. The notebook will run on CPU, but segmentation may be slow.")

print(f"✅ NumPy version: {np.__version__}")

ffmpeg_path = shutil.which("ffmpeg")
print(f"✅ ffmpeg path: {ffmpeg_path if ffmpeg_path else 'Not found'}")

# Ensure imageio uses the system ffmpeg binary, matching the behavior in main.py.
if ffmpeg_path:
    os.environ["IMAGEIO_FFMPEG_EXE"] = ffmpeg_path
    print(f"✅ IMAGEIO_FFMPEG_EXE set to: {os.environ['IMAGEIO_FFMPEG_EXE']}")

In [ ]:
# ============================================================
# 5. Repository Setup and Editable Install
# ============================================================
#
# Clone the repository, move trainer_app to /content,
# and install it in editable mode so that the vimsam-segmenter
# console command is available.
# ============================================================

import os
import shutil

REPOSITORY_URL = "https://github.com/Ilmars-Viksne/viMSam.git"
REPOSITORY_DIR = "/content/viMSam"
PROJECT_DIR = "/content/trainer_app"

# Clean up any previous copies to make the notebook rerunnable.
if os.path.exists(REPOSITORY_DIR):
    shutil.rmtree(REPOSITORY_DIR)

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

# Clone the repository quietly.
!git clone -q https://github.com/Ilmars-Viksne/viMSam.git /content/viMSam

# Move only the segmentation application to /content.
!mv /content/viMSam/trainer_app /content/trainer_app

# Remove the rest of the repository.
!rm -rf /content/viMSam



In [ ]:
# Install the app package in editable mode.
# This makes the vimsam-segmenter console command available.
%cd /content/trainer_app
!pip install -e .

print(f"✅ Project prepared and installed at: {PROJECT_DIR}")

In [ ]:
# ============================================================
# 6. Verify Package and CLI
# ============================================================
#
# Verify that the package was installed correctly and
# the CLI commands are available.
# ============================================================

!vimsam-segmenter --help

print("\n" + "="*60 + "\n")

!python /content/trainer_app/main.py --help

In [ ]:
# ============================================================
# 8. Inspect Project Structure
# ============================================================

!find /content/trainer_app -maxdepth 3 -type f | sort | head -100

In [ ]:
# ============================================================
# Run Workflow: Fine-tuning of micro-SAM models on raw frames
# ============================================================

!vimsam-trainer \
  --images "data/training/raw_frames/" \
  --masks "data/training/masks/" \
  --out "data/models/cell_model_v1/" \
  --workflow "raw_frames" \
  --model "vit_b" \
  --epochs "1" \
  --batch-size "1" \
  --raw-width "1024" \
  --raw-height "1024" \
  --preprocessing-method "fixed_16bit" \
  --patch-shape "1024,1024" \
  --min-size "25" \
  --min-instances-per-patch "1" \
  --n-objects-per-batch "1" \
  --num-workers "0" \
  --device "cuda"



In [ ]:
# ============================================================
# Optional: Package Outputs
# ============================================================
#
# These commands are optional and preserve the behavior of the
# commented-out commands in the original script.
# ============================================================

# Zip only generated models.
#!zip -r models.zip /content/trainer_app/data/models

# Zip some files.
#!zip -r model_best.zip /content/trainer_app/data/models/cell_model_v1/checkpoints/cell_model_v1/best.pt
#!zip -r model_latest.zip /content/trainer_app/data/models/cell_model_v1/checkpoints/cell_model_v1/latest.pt
#!zip -r model_training_summary.zip /content/trainer_app/data/models/cell_model_v1/training_summary.json
#!zip -r model_logs.zip /content/trainer_app/data/models/cell_model_v1/logs/cell_model_v1

# Zip the full segmentation application.
#!zip -r trainer_app.zip /content/trainer_app

In [ ]:
# For the initial one-epoch test:
'''
!vimsam-trainer \
  --images "data/training/raw_frames" \
  --masks "data/training/masks" \
  --out "data/models/cell_model_test" \
  --workflow raw_frames \
  --model vit_b \
  --epochs 1 \
  --batch-size 1 \
  --raw-width 1024 \
  --raw-height 1024 \
  --preprocessing-method fixed_16bit \
  --patch-shape 1024,1024 \
  --min-size 25 \
  --min-instances-per-patch 1 \
  --n-objects-per-batch 1 \
  --num-workers 0 \
  --device cuda
'''

# After this starts training, try the intended patch size:
'''
!vimsam-trainer \
  --images "data/training/raw_frames" \
  --masks "data/training/masks" \
  --out "data/models/cell_model_v1" \
  --workflow raw_frames \
  --model vit_b \
  --epochs 50 \
  --batch-size 2 \
  --raw-width 1024 \
  --raw-height 1024 \
  --preprocessing-method fixed_16bit \
  --patch-shape 512,512 \
  --min-size 25 \
  --min-instances-per-patch 1 \
  --n-objects-per-batch 1 \
  --num-workers 0 \
  --device cuda
'''



In [ ]:
# Use a full-frame patch first:
'''
!vimsam-trainer \
  --images "data/training/raw_frames" \
  --masks "data/training/masks" \
  --out "data/models/cell_model_test" \
  --workflow raw_frames \
  --model vit_b \
  --epochs 1 \
  --batch-size 1 \
  --raw-width 1024 \
  --raw-height 1024 \
  --preprocessing-method fixed_16bit \
  --patch-shape 1024,1024 \
  --min-size 5 \
  --min-instances-per-patch 1 \
  --n-objects-per-batch 1 \
  --num-workers 0 \
  --device cuda
'''

# If that succeeds, change only:
'''
--patch-shape 512,512
'''

